# Day 5 - Deep Learning Projects

## Objectives
- Build a CNN for Cats vs Dogs image classification.
- Apply data augmentation and dropout.
- Build a CNN for IMDB sentiment classification.
- Evaluate model performance and visualize metrics.


## Part 1: Cats vs Dogs Image Classification

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report


In [ ]:
import zipfile, shutil, os
from pathlib import Path

# cherche data/.../*.zip en partant du répertoire courant et en remontant
cwd = Path().resolve()
zip_path = None
for p in [cwd] + list(cwd.parents):
    data_dir = p / "data"
    if data_dir.exists():
        # cherche un zip pertinent (priorité aux noms contenant cat/dog)
        cands = list(data_dir.glob("*cats*.zip")) + list(data_dir.glob("*dogs*.zip")) + list(data_dir.glob("*.zip"))
        if cands:
            zip_path = cands[0]
            break

# fallback path possible (chemin absolu dans ton workspace)
if zip_path is None:
    candidate = Path("/Users/fahim/Documents/Formation/TTA/DI_Bootcamp/data/dogs-vs-cats.zip")
    if candidate.exists():
        zip_path = candidate

if zip_path is None:
    raise FileNotFoundError("Aucun fichier .zip trouvé dans un dossier 'data'. Place le zip dans data/ ou ajuste le chemin.")

dest = cwd / "cats_dogs"
if dest.exists():
    print(f"{dest} existe déjà — suppression/skipping possible si tu veux refaire l'extraction.")
else:
    dest.mkdir(parents=True, exist_ok=True)
    print(f"Extraction de {zip_path} → {dest} ...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest)
    print("Extraction terminée.")

# Organisation : si pas de sous-dossiers cats/dogs, et que les images sont directement présentes ou dans un sous-dossier unique,
# on crée cats/ et dogs/ et on déplace selon le préfixe du nom (cat.*, dog.*).
has_cat_dir = any(p.name.lower().startswith("cat") or p.name.lower().startswith("cats") for p in dest.iterdir() if p.is_dir())
has_dog_dir = any(p.name.lower().startswith("dog") or p.name.lower().startswith("dogs") for p in dest.iterdir() if p.is_dir())

if not (has_cat_dir and has_dog_dir):
    cats_dir = dest / "cats"
    dogs_dir = dest / "dogs"
    cats_dir.mkdir(exist_ok=True)
    dogs_dir.mkdir(exist_ok=True)

    moved = 0
    for img in dest.rglob("*.[jJ][pP][gG]"):
        # ignore images already in cats/ or dogs/
        if cats_dir in img.parents or dogs_dir in img.parents:
            continue
        name = img.name.lower()
        if name.startswith("cat"):
            shutil.move(str(img), str(cats_dir / img.name))
            moved += 1
        elif name.startswith("dog"):
            shutil.move(str(img), str(dogs_dir / img.name))
            moved += 1

    print(f"Images déplacées vers cats/ et dogs/: {moved} fichiers.")

print("Prêt. Utilise 'cats_dogs' comme dossier pour flow_from_directory.")

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# Replace 'cats_dogs' with your dataset directory
train_generator = train_datagen.flow_from_directory(
    'cats_dogs',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    'cats_dogs',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)


In [ ]:
model = models.Sequential([
    layers.Input(shape=(128,128,3)),
    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128,activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1,activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)


In [ ]:
loss, accuracy = model.evaluate(val_generator)

print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy)


In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Accuracy')
plt.legend(['Train','Validation'])

plt.subplot(1,2,2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Loss')
plt.legend(['Train','Validation'])

plt.show()


In [ ]:
predictions = model.predict(val_generator)
predictions = (predictions > 0.5).astype(int)

cm = confusion_matrix(
    val_generator.classes,
    predictions[:len(val_generator.classes)]
)

sns.heatmap(cm, annot=True, fmt='d')
plt.title('Confusion Matrix')
plt.show()

print(classification_report(
    val_generator.classes,
    predictions[:len(val_generator.classes)]
))
